# MuLAD on MIMOSA - full experiment grid

Reimplementation of **MuLAD** (Hasan et al., *Multimodal Aggression Detection from Social
Media Memes Exploiting Visual and Textual Features*) applied to the five-way MIMOSA corpus,
alongside the existing MAF replication.

The paper's design is frozen CNN backbones plus a small trainable text branch, joined by
concatenation. Because nothing in the backbone is ever updated, every meme's visual feature
map is a constant - so this notebook computes them **once** and then trains all 39
configurations against that cache. That turns the paper's full grid from hours of repeated
convolution into a few minutes of dense layers.

**Model selection is on validation; the test split is scored once per run.** With 39
configurations, choosing the winner on test would overfit it by construction.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Notebook settings (right panel) -> Accelerator -> GPU T4 x2'

## 2. Locate the attached dataset

Auto-detects the mounted dataset by looking for `Dataset/Img` under `/kaggle/input`, so the
dataset slug does not need hardcoding.

In [ ]:
import os

def find_input_root(target=os.path.join('Dataset', 'Img')):
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.endswith(target):
            return os.path.dirname(os.path.dirname(root))
    return None

INPUT_ROOT = find_input_root()
assert INPUT_ROOT, ('No Dataset/Img found under /kaggle/input. '
                    'Run print(os.listdir("/kaggle/input")) to inspect the real layout.')
print('Dataset mount:', INPUT_ROOT)
print('memes    :', len(os.listdir(INPUT_ROOT + '/Dataset/Img')))
print(sorted(os.listdir(INPUT_ROOT)))

## 3. Dependencies

MuLAD needs far less than MAF: no CLIP, no transformers. `gensim` is required only for
`--embedding selftrained` and `--embedding fasttext` (the paper's 300-d word-vector rows);
`--embedding keras` learns its own 64-d table and needs nothing extra.

In [ ]:
!pip install -q gensim
import gensim
print('gensim', gensim.__version__)

## 4. Writable working copy

`/kaggle/input` is read-only but the scripts create `Features/`, `Outputs/` and
`Saved_Models/`. Only `Scripts/` is copied; the images stay on the read-only mount and are
referenced by absolute path.

In [ ]:
import shutil

ROOT = '/kaggle/working/Replication'
os.makedirs(ROOT, exist_ok=True)
if os.path.isdir(ROOT + '/Scripts'):
    shutil.rmtree(ROOT + '/Scripts')
shutil.copytree(INPUT_ROOT + '/Scripts', ROOT + '/Scripts')

DATASET_PATH  = INPUT_ROOT + '/Dataset'
FEATURES_PATH = ROOT + '/Features'
print('scripts  :', ROOT + '/Scripts')
print('dataset  :', DATASET_PATH, '(read-only)')
print('features :', FEATURES_PATH)

## 5. Splits

In [ ]:
import pandas as pd
for name in ['training_set', 'validation_set', 'testing_set']:
    df = pd.read_csv(DATASET_PATH + '/' + name + '.csv')
    print('%-16s %5d rows  ' % (name, len(df)), dict(df['Label'].value_counts()))

## 6. Cache the frozen visual features

The only GPU-heavy step: one forward pass per backbone over every meme, stored as float16.

| backbone | cached shape | size |
|---|---|---|
| VGG16 | (512, 7, 7) | ~240 MB |
| VGG19 | (512, 7, 7) | ~240 MB |
| ResNet50 | (2048, 7, 7) | ~970 MB |

Both of the paper's poolings derive from this one cache - `flatten` for the fusion models
(Sec. 4.2) and global average pooling for the visual baselines (Sec. 5.1). Re-running is
cheap: existing caches are skipped unless `--overwrite` is passed.

In [ ]:
%cd {ROOT}/Scripts
!python mulad_features.py --dataset "{DATASET_PATH}" --out "{FEATURES_PATH}" --models vgg16 vgg19 resnet50

## 7. Smoke test

The whole path - data, cache join, model, metrics, `Outputs/` - on a handful of memes.
**Not a result**: 150 memes for 2 epochs against a 20% chance baseline. Its files are tagged
`_subset150` so they cannot overwrite a real run.

In [ ]:
%cd {ROOT}/Scripts
!python mulad_main.py --dataset "{DATASET_PATH}" --features "{FEATURES_PATH}" \
    --modality multimodal --text_model cnn --backbone vgg16 --embedding keras \
    --subset 150 --epochs 2 --patience 0

## 8. The proposed model

**CNN + VGG16** is the configuration the paper names MuLAD. Run it alone first, with full
output, before committing to the grid.

In [ ]:
%cd {ROOT}/Scripts
!python mulad_main.py --dataset "{DATASET_PATH}" --features "{FEATURES_PATH}" \
    --modality multimodal --text_model cnn --backbone vgg16 --embedding keras \
    --run_name mulad_proposed --save_model

## 9. The full grid

39 runs: 9 textual baselines (Table 3), 3 visual baselines (Table 4) and 27 multimodal
models (Table 5). `--skip_existing` makes the cell resumable if the session drops.

For a fast 13-run pass using only the learned 64-d table, add `--embeddings keras`.

In [ ]:
%cd {ROOT}/Scripts
!python mulad_grid.py --dataset "{DATASET_PATH}" --features "{FEATURES_PATH}" --skip_existing

## 10. Results

Every MuLAD run against the MAF replication and both papers' published numbers.

Neither published row is a like-for-like target. MuLAD's 0.738 is a **binary** task on a
different 1,718-meme corpus; MAF's 0.742 is this five-way corpus and is the meaningful
comparison.

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in sorted(glob.glob(ROOT + '/Outputs/results_*.json')):
    r = json.load(open(path, encoding='utf-8'))
    if r.get('hyperparameters', {}).get('subset'):
        continue                      # smoke tests are not results
    rows.append({
        'run': r['run_name'],
        'framework': r.get('framework', 'MAF'),
        'modality': r.get('modality', 'multimodal'),
        'A': round(r['accuracy'], 3),
        'WF': round(r['weighted_f1'], 3),
        'macroF1': round(r['macro_f1'], 3),
        'MMAE': round(r['mmae'], 3),
        'degenerate': r.get('degenerate', False),
    })

table = pd.DataFrame(rows).sort_values('WF', ascending=False)
print(table.to_string(index=False))

print()
print('Published reference points')
print('  MAF   (Ahsan et al. 2024, 5-way MIMOSA) : A 0.741  WF 0.742  MMAE 0.645')
print('  MuLAD (Hasan et al., BINARY AMemD)      : A 0.778  WF 0.738')
print('  MuLAD best TEXT-ONLY baseline (binary)  :          WF 0.889')

## 11. Did fusion help?

The paper's central negative result: its multimodal model scored **below its own text-only
baseline** (0.738 vs 0.889 weighted F1). This checks whether that holds on MIMOSA, whose
classes are far better balanced than AMemD's 92/8 split.

In [ ]:
best = {}
for modality in ['text', 'visual', 'multimodal']:
    part = table[(table['framework'] == 'MuLAD') & (table['modality'] == modality)]
    if not part.empty:
        best[modality] = part.iloc[0]
        print('best %-11s %-40s WF %.3f' % (modality, part.iloc[0]['run'], part.iloc[0]['WF']))

if 'text' in best and 'multimodal' in best:
    delta = best['multimodal']['WF'] - best['text']['WF']
    print()
    print('Concatenation fusion %s by %+.3f WF1 against the best text-only model.'
          % ('HELPED' if delta > 0 else 'HURT', delta))
    print('Paper on AMemD: -0.151 (0.738 vs 0.889).')

deg = table[table['degenerate']]
print()
print('%d of %d runs predicted a single class for every test meme.' % (len(deg), len(table)))
if len(deg):
    print(deg[['run', 'A', 'WF']].to_string(index=False))

## 12. Confusion matrix

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

RUN = 'mulad_proposed'
r = json.load(open(ROOT + '/Outputs/results_' + RUN + '.json', encoding='utf-8'))
plt.figure(figsize=(6, 5))
sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=r['target_names'], yticklabels=r['target_names'])
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('%s  -  WF1 %.3f' % (RUN, r['weighted_f1']))
plt.tight_layout(); plt.show()

## 13. Save the outputs

`/kaggle/working` is wiped when the session ends. Commit the notebook, or download this
archive, before closing.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/mulad_outputs', 'zip', ROOT + '/Outputs')
print('wrote /kaggle/working/mulad_outputs.zip')
print(sorted(os.listdir(ROOT + '/Outputs'))[:50])